*Prof. Stefano Diciotti*  
*University of Bologna*  

---

**Academic Year:** 2025/2026  
**Student Name:** Sergio  
**Degree Program:** ____________________  
**Submission Date:** ____________________

# AI for Medicine Project - Clinically Staged Heart Disease Prediction

This project studies the Cleveland Heart Disease dataset as a binary classification problem. The target is the presence or absence of heart disease.

The main idea is not only to train a classifier, but to answer a clinically meaningful question:

**How much predictive performance is gained when progressively more specialised diagnostic information is added to routinely available cardiovascular risk factors?**

This follows the central principle of the AI for Medicine course: **the first step is to study the medical issue**, then design a reproducible and leakage-safe machine learning pipeline.

# Project Overview

The workflow is organised into six main steps:

1. **Medical background**  
   Understand what clinical problem the dataset represents and why staged information availability matters.

2. **Dataset loading and quality control**  
   Check sample size, variables, missing values, duplicated rows, target distribution, and clinically plausible ranges.

3. **Exploratory data analysis**  
   Compare relevant clinical variables between patients with and without heart disease.

4. **Clinical staging of features**  
   Build incremental feature sets that resemble a possible diagnostic pathway: routine risk factors, symptoms/resting ECG, exercise stress test, and advanced diagnostic measurements.

5. **Leakage-safe machine learning pipeline**  
   Use a scikit-learn `Pipeline` and `ColumnTransformer` so that imputation, scaling, one-hot encoding, model fitting, and hyperparameter tuning are contained inside the validation procedure.

6. **Model evaluation and interpretation**  
   Estimate performance with nested cross-validation on the development set, then use the locked test set only once for final evaluation.

# 1. Medical Background

Heart disease is one of the leading causes of death worldwide. Early identification of patients with possible coronary heart disease can support clinical decision-making, triage, and prioritisation for further diagnostic testing.

In real clinical practice, not all information is available at the same time. A physician may initially know only age, sex, blood pressure, cholesterol, and fasting blood sugar. Later, symptoms, ECG findings, exercise stress test results, and more advanced diagnostic measurements may become available.

For this reason, the project is framed as a staged prediction problem rather than a single generic classification exercise.

## 1.1 Clinical Motivation of the Staged Design

The staged design is clinically relevant because advanced diagnostic tests may be more informative, but they can also be more expensive, invasive, or less available.

The project therefore asks:

- Can routine risk factors provide useful screening information?
- How much is gained by adding symptoms and resting ECG?
- Does the exercise stress test add a relevant improvement?
- Are advanced diagnostic variables driving most of the predictive performance?

This makes the analysis closer to a clinical workflow and less like a generic Kaggle-style model comparison.

# 2. Initial Setup

The code is intentionally simple and close to the style used in the course notebooks. Helper functions are stored in `src/` to avoid repeating code, while this notebook contains the full narrative and all experimental steps.

In [ ]:
# Colab/local setup
# This cell makes the notebook runnable both in Google Colab and locally.

import os
import sys
import subprocess
import importlib.util
import warnings
from pathlib import Path

REPO_URL = "https://github.com/SergioPascualRedondo/Proyect_SP_AI_FOR_MEDICINE.git"
REPO_DIR = Path("/content/Proyect_SP_AI_FOR_MEDICINE")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # When this notebook is opened from GitHub in Colab, the repository files
    # are not automatically available in /content. We clone the repository once.
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    os.chdir(REPO_DIR)
    PROJECT_ROOT = REPO_DIR
else:
    # Local execution: support running from the repository root or from notebooks/.
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        PROJECT_ROOT = cwd.parent
    elif (cwd / "src").exists() and (cwd / "data").exists():
        PROJECT_ROOT = cwd
    else:
        # Fallback for IDEs that execute cells from a different working directory.
        candidates = [cwd, *cwd.parents]
        PROJECT_ROOT = next((c for c in candidates if (c / "src").exists() and (c / "data").exists()), cwd)

sys.path.insert(0, str(PROJECT_ROOT))

# Colab usually already provides these packages. Install only if something is missing.
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}
missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

from src.config import SEED, STAGE_FEATURES, TARGET
from src.data import load_heart_data, data_quality_report, split_features_target
from src.models import build_pipeline, candidate_models
from src.validation import make_locked_split, nested_cv_stage
from src.evaluation import classification_metrics, threshold_metrics_table
from src.visualization import (
    save_figure,
    plot_target_distribution,
    plot_numeric_by_condition,
    plot_numeric_histograms,
    plot_categorical_proportions,
    plot_correlation_heatmap,
    plot_target_correlations,
    plot_stage_sizes,
    plot_cv_auc_by_stage,
    plot_cv_auc_lines,
    plot_threshold_tradeoff,
)

np.random.seed(SEED)
sns.set_theme(style="whitegrid")

FIG_EDA = PROJECT_ROOT / "results" / "figures" / "eda"
FIG_CV = PROJECT_ROOT / "results" / "figures" / "cv"
FIG_TEST = PROJECT_ROOT / "results" / "figures" / "final_test"
TAB_EDA = PROJECT_ROOT / "results" / "tables" / "eda"
TAB_CV = PROJECT_ROOT / "results" / "tables" / "cv"
TAB_TEST = PROJECT_ROOT / "results" / "tables" / "final_test"

for folder in [FIG_EDA, FIG_CV, FIG_TEST, TAB_EDA, TAB_CV, TAB_TEST]:
    folder.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

# 3. Dataset Loading

The project uses the cleaned Cleveland Heart Disease CSV. The original UCI dataset contains several heart disease databases, but the Cleveland subset is the most commonly used for machine learning examples.

In this version, the target variable is already binary:

- `condition = 0`: absence of heart disease
- `condition = 1`: presence of heart disease

In [ ]:
df = load_heart_data()
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

## 3.1 Variable Meaning

| Variable | Clinical meaning | Type used in the pipeline |
|---|---|---|
| `age` | Patient age | Numeric |
| `sex` | Patient sex | Binary |
| `cp` | Chest pain type | Categorical |
| `trestbps` | Resting blood pressure | Numeric |
| `chol` | Serum cholesterol | Numeric |
| `fbs` | Fasting blood sugar > 120 mg/dl | Binary |
| `restecg` | Resting ECG results | Categorical |
| `thalach` | Maximum heart rate achieved | Numeric |
| `exang` | Exercise-induced angina | Binary |
| `oldpeak` | ST depression induced by exercise | Numeric |
| `slope` | Slope of peak exercise ST segment | Categorical |
| `ca` | Number of major vessels coloured by fluoroscopy | Numeric/ordinal |
| `thal` | Thallium stress test result | Categorical |
| `condition` | Heart disease absence/presence | Binary target |

# 4. Data Quality Control

Before modelling, the dataset must be inspected. This is part of the reproducibility workflow: the report should make clear what data were used and whether any cleaning decisions were necessary.

In [ ]:
report = data_quality_report(df)

print("Rows:", report["n_rows"])
print("Columns:", report["n_columns"])
print("Duplicated rows:", report["duplicated_rows"])
print("\nMissing values per column:")
print(report["missing_values"])
print("\nQuestion mark values per column:")
print(report["question_mark_values"])
print("\nTarget counts:")
print(report["target_counts"])

In [ ]:
df.describe().T

## 4.1 Preliminary Data Quality Interpretation

This cleaned version has no missing values, no `?` values, and no duplicated rows. This does not make the project trivial: the main methodological challenge is not data cleaning, but **valid clinical staging, leakage-safe preprocessing, and robust validation on a small dataset**.

Potentially extreme values such as high cholesterol or high resting blood pressure should not be removed automatically, because they may represent real high-risk patients.

# 5. Exploratory Data Analysis

The EDA is not used to tune the model. Its purpose is to understand the medical variables, inspect class balance, and identify clinically plausible patterns.

In [ ]:
fig, ax = plot_target_distribution(df)
plt.show()

In [ ]:
print((df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2))

The target is reasonably balanced. Therefore, the project can focus on clinical staging, validation, and threshold behaviour rather than class-imbalance corrections.

## 5.0 Visual Summary of the Dataset

The following figures make the dataset more readable before modelling. The goal is not to select features from the full dataset, but to understand the clinical variables and class structure.

In [ ]:
fig, axes = plot_numeric_histograms(df, ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"])
save_figure(fig, FIG_EDA / "numeric_distributions_by_condition.png")
plt.show()

In [ ]:
fig, axes = plot_categorical_proportions(df, ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"])
save_figure(fig, FIG_EDA / "categorical_proportions_by_condition.png")
plt.show()

## 5.1 Numeric Variables by Condition

The following plots compare relevant numeric measurements between patients without and with heart disease.

In [ ]:
for column in ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]:
    fig, ax = plot_numeric_by_condition(df, column)
    plt.show()

## 5.2 Categorical Variables by Condition

For categorical and binary variables, proportions are more informative than raw counts.

In [ ]:
categorical_to_plot = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

for column in categorical_to_plot:
    table = pd.crosstab(df[column], df[TARGET], normalize="columns") * 100
    print("\n", column)
    display(table.round(1))

## 5.3 Simple Correlation Check

This check is descriptive only. It should not be interpreted as causal evidence, and categorical encodings can make correlations harder to interpret. Still, it gives a quick first view of variables associated with the target.

In [ ]:
correlations = df.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
correlations.to_frame("correlation_with_condition")

In [ ]:
fig, ax = plot_correlation_heatmap(df)
save_figure(fig, FIG_EDA / "correlation_heatmap.png")
plt.show()

In [ ]:
fig, ax = plot_target_correlations(df)
save_figure(fig, FIG_EDA / "target_correlations.png")
plt.show()

# 6. Clinical Feature Stages

The stages are defined before model evaluation. This prevents adapting the experimental design after observing performance.

In [ ]:
for stage_name, features in STAGE_FEATURES.items():
    print(stage_name)
    print("Number of features:", len(features))
    print(features)
    print()

In [ ]:
fig, ax = plot_stage_sizes(STAGE_FEATURES)
save_figure(fig, FIG_EDA / "clinical_stage_sizes.png")
plt.show()

| Stage | Features added | Clinical interpretation |
|---|---|---|
| Stage 1 | `age`, `sex`, `trestbps`, `chol`, `fbs` | Routine cardiovascular risk factors |
| Stage 2 | + `cp`, `restecg` | Symptoms and resting ECG |
| Stage 3 | + `thalach`, `exang`, `oldpeak`, `slope` | Exercise stress test information |
| Stage 4 | + `ca`, `thal` | More advanced diagnostic measurements |

The comparison across stages is the central experiment of the project.

# 7. The Machine Learning Pipeline

This is the most important methodological part of the project.

Following the course lectures on pipelines, data leakage, and the harmonization paper provided in the course material, every operation that learns parameters from the data must be inside the training process. In this tabular project, this includes:

- median imputation for numeric variables;
- most-frequent imputation for binary/categorical variables;
- standardization of numeric variables;
- one-hot encoding of categorical variables;
- classifier training;
- hyperparameter tuning.

If scaling or encoding were fitted on the full dataset before cross-validation, information from validation/test folds would leak into training.

## 7.1 Inspecting One Complete Pipeline

The pipeline below is not fitted yet. It only shows the structure that will later be fitted inside cross-validation.

In [ ]:
models = candidate_models()
example_stage_name = "Stage 4 - Advanced diagnostic tests"
example_features = STAGE_FEATURES[example_stage_name]
example_model = models["Logistic Regression"]["estimator"]

example_pipeline = build_pipeline(example_features, example_model)
example_pipeline

## 7.2 Why `Pipeline` Matters

The `Pipeline` object guarantees that the following happens separately in each training fold:

1. The imputer learns medians/modes only from the training fold.
2. The scaler learns mean and standard deviation only from the training fold.
3. The one-hot encoder learns categories only from the training fold.
4. The classifier is fitted only on the transformed training fold.
5. The validation fold is transformed using the parameters learned from the training fold.

This is the same principle as the harmonizer transformer discussed in the course: fit preprocessing on training data only, then apply it to unseen data.

# 8. Validation Strategy

The validation scheme follows three rules:

1. A stratified test set is locked before model selection.
2. Nested cross-validation is performed only on the development set.
3. The final test set is used once, after choosing the final modelling strategy.

This avoids optimistic bias caused by using the same data for model selection and model evaluation.

In [ ]:
train_df, test_df = make_locked_split(df, test_size=0.2)

print("Development set:", train_df.shape)
print("Locked test set:", test_df.shape)
print("\nDevelopment target distribution:")
print(train_df[TARGET].value_counts(normalize=True).sort_index().round(3))
print("\nLocked test target distribution:")
print(test_df[TARGET].value_counts(normalize=True).sort_index().round(3))

## 8.1 Nested Cross-Validation Design

The outer loop estimates generalization performance. The inner loop selects hyperparameters.

For each clinical stage and model:

- Outer loop: repeated stratified 5-fold cross-validation.
- Inner loop: stratified 5-fold grid search.
- Scoring metric: ROC-AUC.

ROC-AUC is appropriate because it evaluates discrimination across thresholds and is widely used in medical binary classification.

# 9. Model Choice

The primary model is **Logistic Regression**, because it is simple, interpretable, and appropriate for a small tabular medical dataset.

A **Random Forest** is included as a secondary comparator because it can capture nonlinear interactions. However, interpretation should remain focused on the clinical-stage question, not on trying many algorithms.

In [ ]:
for model_name, model_spec in models.items():
    print(model_name)
    print(model_spec["param_grid"])
    print()

# 10. Nested Cross-Validation Experiment

`N_REPEATS` controls the number of repetitions of the outer 5-fold CV. A smaller value is useful while developing the notebook; a larger value gives a more stable final estimate.

In [ ]:
N_REPEATS = 3
all_results = []

for stage_name, stage_features in STAGE_FEATURES.items():
    for model_name, model_spec in models.items():
        print(f"Running: {stage_name} | {model_name}")
        stage_results = nested_cv_stage(
            train_df=train_df,
            stage_name=stage_name,
            stage_features=stage_features,
            model_name=model_name,
            model_spec=model_spec,
            n_repeats=N_REPEATS,
        )
        all_results.append(stage_results)

cv_results = pd.concat(all_results, ignore_index=True)
cv_results.head()

In [ ]:
cv_summary = (
    cv_results
    .groupby(["stage", "model"])[["roc_auc", "accuracy", "balanced_accuracy"]]
    .agg(["mean", "std"])
)
cv_summary.to_csv(TAB_CV / "nested_cv_summary.csv")
cv_summary

In [ ]:
fig, ax = plot_cv_auc_by_stage(cv_results)
save_figure(fig, FIG_CV / "nested_cv_auc_boxplot_by_stage.png")
plt.show()

In [ ]:
fig, ax = plot_cv_auc_lines(cv_results)
save_figure(fig, FIG_CV / "nested_cv_auc_line_by_stage.png")
plt.show()

## 10.1 Interpretation of Cross-Validation Results

This section should be completed after running the experiment. The interpretation should focus on whether performance increases gradually or whether most of the gain appears only after adding advanced diagnostic measurements.

The main comparison is between stages, not only between algorithms.

# 11. Selecting the Final Strategy

The final model should be selected using nested CV results and clinical reasoning. The locked test set remains untouched until this decision is made.

The code below provides a transparent way to select the best average ROC-AUC combination from the development results.

In [ ]:
mean_auc = (
    cv_results
    .groupby(["stage", "model"], as_index=False)["roc_auc"]
    .mean()
    .sort_values("roc_auc", ascending=False)
)
mean_auc

In [ ]:
best_row = mean_auc.iloc[0]
selected_stage = best_row["stage"]
selected_model = best_row["model"]
selected_features = STAGE_FEATURES[selected_stage]
selected_spec = models[selected_model]

print("Selected stage:", selected_stage)
print("Selected model:", selected_model)
print("Development ROC-AUC:", round(best_row["roc_auc"], 3))

# 12. Threshold Analysis on Development Data

In the course material, probabilistic classifiers are described as models that output a score or probability. A **threshold** is then needed to convert this score into a binary decision.

The ROC analysis discussed in class is based exactly on this idea: changing the threshold changes the balance between true positives and false positives. In medical applications, this also changes the balance between **sensitivity** and **specificity**.

For this reason, the project does not blindly assume that 0.5 is the only possible threshold. Instead, several thresholds are evaluated on out-of-fold predictions from the development set. The final threshold should be chosen before touching the test set and should be justified clinically, for example by prioritising sensitivity in a screening setting or specificity in a confirmatory setting.

In [ ]:
X_dev, y_dev = split_features_target(train_df, selected_features)
final_pipeline = build_pipeline(selected_features, selected_spec["estimator"])

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(
    estimator=final_pipeline,
    param_grid=selected_spec["param_grid"],
    cv=inner_cv,
    scoring="roc_auc",
    refit=True,
)

oof_proba = cross_val_predict(
    search,
    X_dev,
    y_dev,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    method="predict_proba",
    n_jobs=1,
)[:, 1]

dev_threshold_table = threshold_metrics_table(
    y_dev,
    oof_proba,
    thresholds=np.round(np.arange(0.1, 1.0, 0.1), 2),
)

dev_threshold_table.to_csv(TAB_CV / "development_threshold_metrics.csv", index=False)

fig, ax = plot_threshold_tradeoff(dev_threshold_table, title="Development threshold analysis")
save_figure(fig, FIG_CV / "development_threshold_tradeoff.png")
plt.show()

dev_threshold_table[["threshold", "sensitivity", "specificity", "precision", "accuracy", "balanced_accuracy"]]

# 13. Final Test Evaluation

The final test set is used only after model selection. Since the threshold is a clinical operating choice, the test set is reported across the same pre-defined threshold grid used in the development analysis. This shows the sensitivity/specificity trade-off without selecting the threshold based on test performance.

In [ ]:
X_test, y_test = split_features_target(test_df, selected_features)

search.fit(X_dev, y_dev)
y_test_score = search.predict_proba(X_test)[:, 1]

test_threshold_table = threshold_metrics_table(
    y_test,
    y_test_score,
    thresholds=dev_threshold_table["threshold"].to_numpy(),
)

test_threshold_table.to_csv(TAB_TEST / "final_test_threshold_metrics.csv", index=False)

fig, ax = plot_threshold_tradeoff(test_threshold_table, title="Final test threshold analysis")
save_figure(fig, FIG_TEST / "final_test_threshold_tradeoff.png")
plt.show()

test_threshold_table[["threshold", "roc_auc", "sensitivity", "specificity", "precision", "accuracy", "balanced_accuracy", "tn", "fp", "fn", "tp"]]

In [ ]:
# Choose a threshold here only after justifying the clinical objective from the development table.
# Example: 0.5 is shown as a reference operating point from the lecture examples, not as an automatic clinical optimum.
report_threshold = 0.5

y_test_pred = (y_test_score >= report_threshold).astype(int)
print(classification_report(y_test, y_test_pred, target_names=["No heart disease", "Heart disease"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, display_labels=["No disease", "Disease"])
plt.title("Final test confusion matrix")
plt.tight_layout()
plt.savefig(FIG_TEST / "final_test_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_test_score)
plt.title("Final test ROC curve")
plt.tight_layout()
plt.savefig(FIG_TEST / "final_test_roc_curve.png", dpi=200, bbox_inches="tight")
plt.show()

# 14. Discussion

The discussion should answer the clinical question, not only report scores.

Important points to discuss:

- Whether routine variables alone are sufficient for rough screening.
- Whether exercise-test variables substantially improve performance.
- Whether advanced diagnostic variables dominate the prediction.
- Whether Logistic Regression is competitive with Random Forest.
- How the small sample size affects uncertainty.
- Why results from this old single-centre dataset should not be considered directly deployable.

# 15. Limitations

The main limitations are:

- small sample size;
- single-centre Cleveland cohort;
- old dataset collected in a different clinical context;
- no external validation cohort;
- already cleaned data, so missing-data robustness is not truly tested;
- categorical encodings require careful interpretation;
- model performance may be optimistic compared with real deployment.

# 16. Ethics and Data Privacy

The dataset is publicly available and de-identified. Even so, the model should be presented only as an academic analysis and not as a diagnostic medical device.

A clinically deployed tool would require external validation, prospective evaluation, calibration analysis, monitoring for dataset shift, and assessment of fairness across patient subgroups.

# 17. Reproducibility Notes

To reproduce the analysis:

1. Use the CSV in `data/raw/heart_cleveland_upload.csv`.
2. Install the dependencies listed in `requirements.txt`.
3. Run this notebook from top to bottom.
4. Keep `SEED = 42` fixed unless explicitly studying random variability.
5. Do not modify the locked test set after it is created.

The important reproducibility point is that all preprocessing is inside the scikit-learn pipeline and therefore follows the validation split.

# 18. References

- Detrano, R. et al. International application of a new probability algorithm for the diagnosis of coronary artery disease. *American Journal of Cardiology*, 1989.
- UCI Machine Learning Repository. Heart Disease Dataset.
- Marzi, C. et al. Harmonization paper provided in the course material, used here for the pipeline/data-leakage principle.
- Diciotti, S. AI for Medicine course material: data leakage, reproducibility, cross-validation, nested cross-validation, and machine learning pipelines.